In [ ]:
!pip install "psycopg[binary]==3.3.6" "python-dotenv==1.2.4"

This one line installs the exact versions of the two libraries we use: `psycopg` (a Python driver, which lets Python talk to PostgreSQL) and `python-dotenv` (which reads our connection settings from the shared `.env` file). Pinning the versions keeps the lesson working months from now.

In [ ]:
# Bring in the tools we need.
import os
from decimal import Decimal
import psycopg
from psycopg import sql
from dotenv import load_dotenv

# Read the connection settings from the shared .env file.
load_dotenv("../../.env")
PGHOST = os.getenv("PGHOST", "localhost")
PGPORT = os.getenv("PGPORT", "5432")
PGUSER = os.getenv("PGUSER", "postgres")
PGPASSWORD = os.getenv("PGPASSWORD") or None
PGDATABASE = os.getenv("PGDATABASE", "university")
PGSSLMODE = os.getenv("PGSSLMODE", "prefer")

We import the libraries and read the connection settings (server address, port, user, password, database name) from the shared `.env` file. Each setting has a safe default, and the password is never written in the code.

In [ ]:
# Connect to the server, create our practice database if it is missing, then open our connection.
conn_args = dict(host=PGHOST, port=PGPORT, user=PGUSER, password=PGPASSWORD, sslmode=PGSSLMODE)
try:
    with psycopg.connect(dbname="postgres", autocommit=True, **conn_args) as admin:
        if not admin.execute("SELECT 1 FROM pg_database WHERE datname = %s", (PGDATABASE,)).fetchone():
            admin.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(PGDATABASE)))
    conn = psycopg.connect(dbname=PGDATABASE, **conn_args)
    print("connected to", PGDATABASE)
except psycopg.OperationalError as err:
    raise SystemExit(f"Could not connect. Is PostgreSQL running, and do the .env settings match it?\n{err}")

We connect twice: first to the built-in `postgres` database to create `university` if it does not exist yet, then to `university` itself. If the server is down or a setting is wrong, we stop with a friendly message instead of a long error. We leave autocommit off, so our changes only become permanent when we call `conn.commit()`.

In [ ]:
# A small helper that runs a query and prints the result as a neat table.
def show(query, params=None):
    """Run a query and print the result as a psql-style table."""
    with conn.cursor() as cur:
        cur.execute(query, params)
        if cur.description is None:              # statements like CREATE or INSERT return no rows
            print(cur.statusmessage)
            return
        cols = [d.name for d in cur.description]
        rows = cur.fetchall()
    fmt = lambda v: "NULL" if v is None else ("t" if v is True else "f" if v is False else str(v))
    cells = [[fmt(v) for v in r] for r in rows]
    numeric = [bool(rows) and all(v is None or (isinstance(v, (int, float, Decimal)) and not isinstance(v, bool)) for v in col)
               for col in zip(*rows)] or [False] * len(cols)
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    print(" | ".join(c.center(w) for c, w in zip(cols, widths)).rstrip())
    print("-+-".join("-" * w for w in widths))
    for r in cells:
        print(" | ".join(v.rjust(w) if numeric[i] else v.ljust(w) for i, (v, w) in enumerate(zip(r, widths))).rstrip())
    print(f"({len(rows)} row{'' if len(rows) == 1 else 's'})")

We will print many query results, so we write this printing code once as a function called `show` and reuse it in every later cell. It runs the query, turns each value into text, lines the columns up, and prints a row count at the bottom.

In [ ]:
# Start clean so we can run this notebook again and again.
conn.execute("DROP TABLE IF EXISTS orders")

# An orders table with a mix of numbers, words, an email and a timestamp.
conn.execute("""
    CREATE TABLE orders (
        order_id    INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        customer_id INTEGER NOT NULL,
        status      TEXT NOT NULL,
        email       TEXT NOT NULL,
        amount      NUMERIC(8,2) NOT NULL,
        created_at  TIMESTAMP NOT NULL
    )
""")

# Let PostgreSQL invent 50,000 rows: 2,000 customers, a few "pending" orders, one order every 10 minutes.
conn.execute("""
    INSERT INTO orders (customer_id, status, email, amount, created_at)
    SELECT g % 2000 + 1,
           CASE WHEN g % 50 = 0 THEN 'pending' WHEN g % 7 = 0 THEN 'cancelled' ELSE 'shipped' END,
           'User' || (g % 2000 + 1) || '@Example.com',
           (g % 500) + 0.99,
           TIMESTAMP '2026-01-01' + g * INTERVAL '10 minutes'
    FROM generate_series(1, 50000) AS g
""")

# Collect statistics so the planner knows what the table looks like.
conn.execute("ANALYZE orders")
conn.commit()
show("SELECT count(*) AS orders, count(DISTINCT customer_id) AS customers FROM orders")

We create 50,000 fake orders with `generate_series`, which counts from 1 to 50,000 and gives us one row per number. This is the smallest table where the difference between a slow and a fast query becomes visible. `ANALYZE` stores statistics about the table (how many rows, how values are spread) so the planner can make good choices. The planner is the part of PostgreSQL that decides *how* to run a query.

In [ ]:
# EXPLAIN shows the plan PostgreSQL chose. We call it many times, so we wrap it in a small function.
def explain(query, buffers=False):
    """Run EXPLAIN ANALYZE on a query and print the plan, without cost numbers."""
    options = "ANALYZE, COSTS OFF, TIMING OFF, SUMMARY OFF" + (", BUFFERS" if buffers else "")
    with conn.cursor() as cur:
        cur.execute(f"EXPLAIN ({options}) {query}")
        for (line,) in cur.fetchall():
            print(line)

`EXPLAIN` shows the plan for a query, and the `ANALYZE` option also *runs* the query and reports what really happened, such as how many rows each step produced. We switch off cost numbers and timings so the output stays short and the same every run. We define a helper because we use it in almost every later step.

In [ ]:
# With no index on customer_id, how does PostgreSQL find customer 42's orders?
explain("SELECT * FROM orders WHERE customer_id = 42", buffers=True)

- **Input:** a query for one customer's orders, on a table with only a primary key index.
- **What happens:** the plan shows a `Seq Scan` (sequential scan): PostgreSQL read every row and threw away the ones that did not match. `Rows Removed by Filter` counts the thrown-away rows, and `Buffers` shows how many 8 kilobyte (KB) pages it touched.
- **Output:** 25 matching rows found by reading all 50,000.
- **Why:** this is the baseline. Reading a plan top to bottom, from the innermost step outwards, is the skill that the rest of the lesson builds on.

It is like finding every page that mentions one name in a book with no index: you read the whole book.

In [ ]:
# A balanced tree (B-tree) index keeps the values sorted so PostgreSQL can jump straight to them.
conn.execute("CREATE INDEX idx_orders_customer ON orders (customer_id)")
conn.execute("ANALYZE orders")
conn.commit()
explain("SELECT * FROM orders WHERE customer_id = 42", buffers=True)

The same query now uses the index (look for `Index Scan` or `Bitmap Index Scan`) and touches far fewer pages, because the index takes PostgreSQL straight to the 25 matching rows. A B-tree index is the default and works for `=`, `<`, `>`, `BETWEEN` and `ORDER BY`. The price is that every insert and update must also update the index.

In [ ]:
# One index over two columns: first status, then created_at.
conn.execute("CREATE INDEX idx_orders_status_created ON orders (status, created_at)")
conn.execute("ANALYZE orders")
conn.commit()

# Uses both columns: PostgreSQL goes to 'shipped', then to one day inside it.
explain("SELECT order_id FROM orders WHERE status = 'shipped' AND created_at >= '2026-06-01' AND created_at < '2026-06-02'")

# Skips the first column. PostgreSQL 18 can still use the index by trying each status in turn (a "skip scan").
explain("SELECT order_id FROM orders WHERE created_at >= '2026-06-01' AND created_at < '2026-06-02'")

- **Input:** an index on `(status, created_at)` and two queries.
- **What happens:**
  - The first query filters on both columns, so PostgreSQL walks to `shipped` and then to one day inside it.
  - The second query leaves out `status`. Before PostgreSQL 18 this index would be almost useless for it. PostgreSQL 18 can run a *skip scan*: because `status` has only a few different values, it searches the date range once for each status.
- **Output:** both plans use the index; the second shows `Index Searches` greater than 1, because PostgreSQL searches the index again for each status.
- **Why:** the order of columns decides which queries an index can help. Put the columns you compare with `=` first.

It is like a phone book sorted by last name, then first name: easy to find "Smith, Ana", hard to find every "Ana" unless there are only a few last names to check.

```mermaid
flowchart LR
  A[Index on status, created_at] --> B{Query filters status?}
  B -->|yes| C[Jump straight in]
  B -->|no, few statuses| D[Skip scan: search each status]
```

In [ ]:
# Replace the plain customer index with one that also carries the amount column.
conn.execute("DROP INDEX idx_orders_customer")
conn.execute("CREATE INDEX idx_orders_customer_amount ON orders (customer_id) INCLUDE (amount)")
conn.commit()

# VACUUM marks pages as "all rows visible", which index-only scans need. It cannot run inside a transaction.
conn.autocommit = True
conn.execute("VACUUM ANALYZE orders")
conn.autocommit = False

# Everything this query needs is inside the index, so it never reads the table.
explain("SELECT customer_id, sum(amount) FROM orders WHERE customer_id = 42 GROUP BY customer_id")

`INCLUDE (amount)` stores an extra column inside the index entries without sorting by it. When a query needs only `customer_id` and `amount`, PostgreSQL can answer from the index alone: an `Index Only Scan` with `Heap Fetches: 0`. `VACUUM` first records which table pages contain only visible rows, so PostgreSQL can trust the index without double-checking the table. This is why a covering index can be the fastest option for a frequent query.

In [ ]:
# Partial index: only 'pending' orders are indexed, so the index stays tiny.
conn.execute("CREATE INDEX idx_orders_pending ON orders (created_at) WHERE status = 'pending'")

# Expression index: indexes lower(email), so case-insensitive lookups can use it.
conn.execute("CREATE INDEX idx_orders_email_lower ON orders (lower(email))")
conn.execute("ANALYZE orders")
conn.commit()

# The query must repeat the WHERE condition for the partial index to apply.
explain("SELECT order_id FROM orders WHERE status = 'pending' ORDER BY created_at LIMIT 5")

# The query must use the same expression, lower(email), for the expression index to apply.
explain("SELECT order_id FROM orders WHERE lower(email) = 'user42@example.com'")

A partial index covers only the rows that match its `WHERE`, so a work queue of 1,000 pending orders gets a tiny index no matter how many shipped orders exist. An expression index stores the *result* of an expression such as `lower(email)`, so searches written the same way can use it. In both cases the query has to match how the index was defined, otherwise PostgreSQL cannot use the index.

In [ ]:
# We no longer need the multicolumn index, so we drop it to give the planner a clearer choice.
conn.execute("DROP INDEX idx_orders_status_created")

# A Block Range Index (BRIN) stores only the smallest and largest value of each group of pages.
conn.execute("CREATE INDEX idx_orders_created_btree ON orders (created_at)")
conn.execute("CREATE INDEX idx_orders_created_brin ON orders USING BRIN (created_at)")
conn.commit()
show("""
    SELECT indexrelid::regclass AS index_name, pg_size_pretty(pg_relation_size(indexrelid)) AS size
    FROM pg_index
    WHERE indexrelid IN ('idx_orders_created_btree'::regclass, 'idx_orders_created_brin'::regclass)
    ORDER BY pg_relation_size(indexrelid) DESC
""")

# Drop the big one and let the small BRIN index answer a one-day range query.
conn.execute("DROP INDEX idx_orders_created_btree")
conn.commit()
explain("SELECT order_id FROM orders WHERE created_at >= '2026-03-01' AND created_at < '2026-03-02'")

- **Input:** `created_at`, which grows with every new order, so rows with similar times sit next to each other on disk.
- **What happens:** a B-tree stores every value, while a BRIN stores just the minimum and maximum for each block range (a run of table pages). PostgreSQL skips every range that cannot hold the dates we ask for.
- **Output:** the BRIN index is tiny compared with the B-tree (24 kB against 1,112 kB on our test server, so your numbers may differ slightly), and it still answers the range query using a `Bitmap Index Scan` followed by a `Bitmap Heap Scan`.
- **Why:** BRIN works well only when the column's order matches the table's physical order, such as time-stamped log or order data, and it can be a thousand times smaller.

```mermaid
flowchart LR
  A[Table pages in time order] --> B[BRIN: min and max per page range]
  B --> C{Range can match?}
  C -->|no| D[Skip these pages]
  C -->|yes| E[Read these pages]
```

In [ ]:
# OFFSET must walk past 40,000 rows just to throw them away.
explain("SELECT order_id, amount FROM orders ORDER BY order_id OFFSET 40000 LIMIT 10")

# Keyset pagination remembers the last id we saw and jumps straight to the next page.
explain("SELECT order_id, amount FROM orders WHERE order_id > 40000 ORDER BY order_id LIMIT 10")

# Both give the same page of results.
show("SELECT order_id, amount FROM orders WHERE order_id > 40000 ORDER BY order_id LIMIT 3")

Look at the `actual rows` of the inner `Index Scan` in each plan: with `OFFSET 40000` PostgreSQL reads 40,010 rows to return 10, and it gets slower the deeper you page. The keyset version starts at `order_id > 40000` and reads only 10. The price is that you can go to "the next page" but not jump to "page 87". Think of a bookmark: you reopen the book where you stopped instead of counting pages from the front each time.

In [ ]:
# What does PostgreSQL know about our columns? n_distinct = how many different values.
show("SELECT attname, n_distinct, correlation FROM pg_stats WHERE tablename = 'orders' AND attname IN ('status', 'customer_id', 'created_at') ORDER BY attname")

# Every index takes space and must be updated on each write. Here is the bill.
show("""
    SELECT indexrelid::regclass AS index_name, pg_size_pretty(pg_relation_size(indexrelid)) AS size
    FROM pg_index WHERE indrelid = 'orders'::regclass
    ORDER BY pg_relation_size(indexrelid) DESC, indexrelid::regclass::text
""")

`pg_stats` is where `ANALYZE` stores what it learned. `n_distinct` is the number of different values (a negative number means a fraction of the row count), and `correlation` runs from -1 to 1 and says how closely the column's order matches the table's physical order: close to 1 is why BRIN worked for `created_at`. Both are estimated from a random sample, so your last digits will differ a little from run to run. The size list is the other half of the story. Each index speeds up reads but takes disk space and slows every `INSERT`, `UPDATE` and `DELETE`, so keep the indexes that earn their place and drop the rest.

In [ ]:
# Close the connection politely. Our tables stay in the database.
conn.close()
print("connection closed")

We close the connection when we are done. The tables and rules we made stay in the database.